In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
from src.utils import setup_logging
from src.financial_statistics import FinancialStatistics
from src.portfolio import PortfolioEngine
from src.optimization import PortfolioOptimizer
from src.simulation import MonteCarloSimulator

In [4]:
# 1. Config & Data Setup
logger = setup_logging()
logger.info("Executing Phase 6: Monte Carlo Simulation Pipeline.")

simple_returns = pd.read_csv("data/processed/simple_returns.csv", index_col="Date", parse_dates=True)
stats_engine = FinancialStatistics()
summary_stats = stats_engine.compute_summary_statistics(simple_returns, simple_returns)
matrices = stats_engine.get_matrices(simple_returns)

2026-07-12 23:42:56 [INFO] utils: Logging infrastructure successfully initialized.
2026-07-12 23:42:56 [INFO] 542838461: Executing Phase 6: Monte Carlo Simulation Pipeline.
2026-07-12 23:42:56 [INFO] financial_statistics: Generating baseline financial statistics profiles...
2026-07-12 23:42:56 [INFO] financial_statistics: Computing asset covariance and correlation matrices.


Writing logs to: /Users/admin/Desktop/Portofolio Analyzer/logs/portfolio_analyzer.log


In [5]:
# 2. Get Max Sharpe metrics from our Phase 7 steps
optimizer = PortfolioOptimizer(rf_rate=0.04)
portfolio_engine = PortfolioEngine()
max_sharpe_weights = optimizer.optimize_max_sharpe(summary_stats["Annualized Return"], matrices["covariance"])

p_ret, p_vol, _ = portfolio_engine.calculate_metrics(
    max_sharpe_weights, 
    summary_stats["Annualized Return"], 
    matrices["covariance"]
)

2026-07-12 23:43:13 [INFO] optimization: Executing Max Sharpe Optimization solver.


In [6]:
# 3. Initialize Simulator Engine
simulator = MonteCarloSimulator()
initial_capital = 10000.0
days_to_simulate = 252
sim_paths_count = 10000

In [7]:
# 4. Generate Stochastic Pricing Scenarios
paths = simulator.simulate_gbm(
    initial_value=initial_capital,
    mu=p_ret,
    sigma=p_vol,
    n_days=days_to_simulate,
    n_simulations=sim_paths_count
)

2026-07-12 23:43:41 [INFO] simulation: Simulating 10000 paths over 252 days (Drift: 0.1975, Vol: 0.1402).


In [8]:
# 5. Extract and Evaluate Terminal End-Value Distributions
terminal_values = paths[-1, :]
percentiles = np.percentile(terminal_values, [5, 25, 50, 75, 95])
probability_of_loss = np.mean(terminal_values < initial_capital) * 100

print(f"\n=== MONTE CARLO FORWARD CAPITALS ANALYSIS (Initial: ${initial_capital:,.0f}) ===")
print(f"Median Expected Ending Portfolio Value (50th Pctl): ${percentiles[2]:,.2f}")
print(f"95% Confidence Interval Floor (5th Pctl Worst Case): ${percentiles[0]:,.2f}")
print(f"95% Optimistic Ceiling (95th Pctl Best Case): ${percentiles[4]:,.2f}")
print(f"Probability of Portfolio Ending with an Absolute Capital Loss: {probability_of_loss:.2f}%")


=== MONTE CARLO FORWARD CAPITALS ANALYSIS (Initial: $10,000) ===
Median Expected Ending Portfolio Value (50th Pctl): $12,071.42
95% Confidence Interval Floor (5th Pctl Worst Case): $9,493.27
95% Optimistic Ceiling (95th Pctl Best Case): $15,199.86
Probability of Portfolio Ending with an Absolute Capital Loss: 9.44%


In [9]:
from src.visualization import FinancialVisualizer
visualizer = FinancialVisualizer()
visualizer.plot_monte_carlo_results(paths)
print("Simulation visualizers archived inside images/ folder.")

2026-07-12 23:45:01 [INFO] visualization: Generating Monte Carlo visualization layout.
2026-07-12 23:45:01 [INFO] visualization: Monte Carlo simulation chart exported to: /Users/admin/Desktop/Portofolio Analyzer/images/monte_carlo_simulation.png


Simulation visualizers archived inside images/ folder.
